In [3]:
import pandas as pd
import numpy as np
import pickle
import re
import sys

from src.neuralnet import NeuralNetwork
from src.numpy_text_features import transform_tfidf

class NeuralNetworkTracked(NeuralNetwork):
    pass
sys.modules["__main__"].NeuralNetworkTracked = NeuralNetworkTracked

PATH_DATASET_TESTE = "data/subm2.csv"
df_teste = pd.read_csv(PATH_DATASET_TESTE, sep=None, engine="python")

# Normalizar colunas para evitar problemas de BOM/capitalização
df_teste.columns = [str(c).replace("\ufeff", "").strip() for c in df_teste.columns]
cols_lower = {c.lower(): c for c in df_teste.columns}
if "id" in cols_lower and "ID" not in df_teste.columns:
    df_teste = df_teste.rename(columns={cols_lower["id"]: "ID"})
if "text" in cols_lower and "Text" not in df_teste.columns:
    df_teste = df_teste.rename(columns={cols_lower["text"]: "Text"})
if "Text" not in df_teste.columns:
    raise ValueError("Coluna 'Text' não encontrada no ficheiro de teste.")
if "ID" not in df_teste.columns:
    df_teste.insert(0, "ID", np.arange(1, len(df_teste) + 1, dtype=np.int64))
id_like = [c for c in df_teste.columns if c.lower() == "id"]
if len(id_like) > 1:
    keep = id_like[0]
    df_teste = df_teste.drop(columns=id_like[1:])
    if keep != "ID":
        df_teste = df_teste.rename(columns={keep: "ID"})

with open("modelo_numpy_artefactos.pkl", "rb") as f:
    artefactos = pickle.load(f)

le = artefactos["label_encoder"]
vocab = artefactos["vocab"]
idf = artefactos["idf"]
net_np = artefactos["model"]

def clean_text_np(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.strip()

texts_clean = df_teste["Text"].apply(clean_text_np).tolist()
X_teste_tfidf = transform_tfidf(texts_clean, vocab, idf)
preds_np_probs = net_np.predict(X_teste_tfidf)
preds_np_idx = np.argmax(preds_np_probs, axis=1)

df_teste["Labels"] = le.inverse_transform(preds_np_idx)

NOME_FICHEIRO_SAIDA = "subm2-g14-MEI-A.csv"
df_saida = df_teste[["ID", "Text", "Labels"]]
df_saida.to_csv(NOME_FICHEIRO_SAIDA, index=False, sep=";")

print(f"Previsões concluídas e guardadas em {NOME_FICHEIRO_SAIDA}")
df_saida.head()

Previsões concluídas e guardadas em subm2-g14-MEI-A.csv


,ID,Text,Labels
0,D2-101,Microbial mats of coexisting bacteria and arch...,Human
1,D2-102,The origin of life on Earth remains a complex ...,OpenAI
2,D2-103,Estimates of the time at which life arose on E...,Human
3,D2-104,Life on Earth emerged roughly 3.8-4 billion ye...,Google
4,D2-105,Black holes predominantly form from the catast...,Google
